# Imports

In [1]:
import cda2
#import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from datetime import datetime, timedelta
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use.

In [4]:
#api.start_spark(n_executors=100, config=config)
api.start_spark(n_executors=400)

https://artifacts.mitre.org/artifactory/java-libs-release added as a remote repository with the name: repo-1
https://dali.mitre.org/nexus/content/repositories/mitre-caasd-releases added as a remote repository with the name: repo-2
https://dali.mitre.org/nexus/content/repositories/external-releases added as a remote repository with the name: repo-3
Ivy Default Cache set to: /home/rchong/.ivy2/cache
The jars for the packages stored in: /home/rchong/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
graphframes#graphframes added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-57574562-6472-46b6-b23b-4bd4f4e37667;1.0
	confs: [default]


:: loading settings :: url = jar:file:/devel/data_access/software/tdp-jupyter/poetry/cache/virtualenvs/python39-QwwvzYkJ-py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in repo-1
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in repo-1
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central
	found org.ejml#ejml-core;0.43.1 in central
	found org.ejml#ejml-ddense;0.43.1 in central
	found com.esri.geometry#esri-geometry-api;2.2.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.9.6 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.14 in central
	found com.uber#h3;4.1.1 in central
	found org.apache.spark#spark-avro_2.12;3.5.1 in central
	found org.tuka

# Define global variables and functions

In [5]:
airports = [
    "KADW",
    "KATL",
    "KBOS",
    "KBWI",
    "KCLT",
    "KDCA",
    "KDEN",
    "KDFW",
    "KDTW",
    "KEWR",
    "KFLL",
    "KIAD",
    "KIAH",
    "KJFK",
    "KLAS",
    "KLAX",
    "KLGA",
    "KMCO",
    "KMDW",
    "KMEM",
    "KMIA",
    "KMSP",
    "KORD",
    "KPHL",
    "KPHX",
    "KSAN",
    "KSDF",
    "KSEA",
    "KSFO",
    "KSLC",
    "KTPA",
    "PANC",
    "PHNL",
]

In [6]:
%run ./shared_variables.ipynb

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [7]:
#year0 = "2025"
year1 = str(int(year0) + 1)

In [8]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}
print(dates)

{'start_date': '2025-01-01', 'end_date': '2026-01-01'}


In [9]:
print("retrieving procedures: ", datetime.now())

retrieving procedures:  2026-03-31 20:00:42.613572


# Retrieve SIDs and STARs for the airports of interest

<li>only get SID and STAR procedure types. (procedure_type = "APPROACH" is ignored)</li>
<li>create a unique <i>grouping</i> column.</li>
</br>
this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [10]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .select(
        "dtpp_procedure_name",
        "procedure_name",
        "transition_name",
        "procedure_type",
        "transition_type",
        F.col("airport.icao_region").alias("airport_icao_region"),
        F.col("airport.name").alias("airport"),
        F.explode("legs").alias("leg"),
        F.col("metadata.effective_end_date").alias("end_date")
    )
    .withColumn("fix_name", F.col("leg.path_terminator.identification.name"))
    .withColumn("fix_icao_region", F.col("leg.path_terminator.identification.icao_region"))
    .withColumn("seq", F.col("leg.sequence_number").alias("seq"))
    .withColumn("latitude", F.col("leg.path_terminator.latitude"))
    .withColumn("longitude", F.col("leg.path_terminator.longitude"))
    .withColumn("magnetic_variation", F.col("leg.path_terminator.magnetic_variation.modeled"))
    .withColumn("speed_description", F.col("leg.speed_limit.descriptor"))
    .withColumn("speed_limit", F.col("leg.speed_limit.limit"))
    .withColumn("speed_altitude", F.col("leg.speed_limit.altitude"))
    .withColumn("altitude_description", F.col("leg.altitude_limits.description"))
    .withColumn("altitude_value1", F.col("leg.altitude_limits.value1"))
    .withColumn("altitude_value2", F.col("leg.altitude_limits.value2"))
    .withColumn("grouping", F.concat("procedure_name", F.lit("_"), "transition_name", F.lit("_"), "seq"))
    .filter(F.col("procedure_type").isin("SID", "STAR"))
    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))
    .drop("leg")
    .orderBy("grouping")
    .persist()
)

df_procedures = (
    df_procedures
    .filter(F.col("grouping").isNotNull())
    .filter(F.col("transition_name").isNotNull())
    .filter(F.col("fix_name").isNotNull())
    .orderBy("grouping")
    .persist()
)

#df_procedures.count()

Multiple versions found: 3.1.71, 3.1.73, 3.1.74, 3.1.75, 3.1.76, 3.1.77, 3.1.79, 3.1.80
                                                                                

## keep only the newest update

In [11]:
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_procedures_newest = (df_procedures
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("grouping")
)

df_procedures_newest.count()

58047

## output procedures (SIDs and STARs) partioned by airport

In [12]:
print("starting airport procedures: ", datetime.now())

starting airport procedures:  2026-03-19 14:38:21.190434


In [13]:
(
    df_procedures_newest
        .select(
            "dtpp_procedure_name",
            "procedure_name",
            "procedure_type",
            "transition_name",
            "transition_type",
            "fix_name",
            "fix_icao_region",
            "seq",
            "latitude",
            "longitude",
            "magnetic_variation",
            "airport",
            "speed_description",
            "speed_limit",
            "speed_altitude",
            "altitude_description",
            "altitude_value1",
            "altitude_value2"
        )
        .repartition("airport")
        .write.option("header",True).partitionBy(["airport"])
        .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="overwrite")
)

In [14]:
print("completed airport procedures: ", datetime.now())

completed airport procedures:  2026-03-19 14:39:14.069874


## output all north american procedure fixes
<li>add a column that concats all the procedures that use the fix</li>
<li>add a column that concats the procedure types that use the fix</li>

In [15]:
print("starting aggregate procedures: ", datetime.now())

starting aggregate procedures:  2026-03-19 14:39:14.077427


In [16]:
(
    df_procedures_newest.groupBy("fix_name", "fix_icao_region", "latitude", "longitude", "magnetic_variation")
        .agg(F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_using_fix"), F.concat_ws(":", F.collect_set("procedure_type")).alias("procedure_types_using_fix"))
        .orderBy("fix_name")
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="append")
)

In [17]:
print("completed aggregate procedures: ", datetime.now())

completed aggregate procedures:  2026-03-19 14:39:17.473559
